# MAX Graph Compiler ⚙️

> Translates the `LogicalPlan` into a `max.graph.Graph` using pure MAX ops.

In [ ]:
#| default_exp compiler

In [ ]:
#| export
from typing import Any, Dict, List, Union
import pyarrow as pa
import pyarrow.compute as pc
import numpy as np
from max import engine, driver
from max.graph import Graph, TensorType, ops
from max.graph.type import DType, DeviceRef
from mxframe.lazy_expr import Expr
from mxframe.lazy_frame import LogicalPlan, Scan, Filter, Project, Aggregate, Sort, Limit, Distinct, Join

## The `GraphCompiler` Class 🛠️

This class traverses the `LogicalPlan` and builds a MAX Graph. It maps our `Expr` nodes to pure `max.graph.ops`.

In [ ]:
#| export
# ── dtype helpers ────────────────────────────────────────────────────────

_NP_TO_MAX = {
    np.dtype("int32"):   DType.int32,
    np.dtype("int64"):   DType.int64,
    np.dtype("float32"): DType.float32,
    np.dtype("float64"): DType.float64,
    np.dtype("bool"):    DType.bool,
}

def _max_dtype(arr) -> DType:
    """Map a numpy dtype to the corresponding MAX DType."""
    return _NP_TO_MAX.get(arr.dtype, DType.float32)

_PC_CMP_OPS = {
    "gt":  pc.greater,
    "ge":  pc.greater_equal,
    "lt":  pc.less,
    "le":  pc.less_equal,
    "eq":  pc.equal,
    "and": pc.and_,
    "or":  pc.or_,
}

_CMP_OPS = {
    "gt": ops.greater,
    "ge": ops.greater_equal,
    "lt": lambda a, b: ops.greater(b, a),
    "le": lambda a, b: ops.greater_equal(b, a),
    "eq": ops.equal,
}


def _is_gpu_max_compatible(acc: driver.Accelerator) -> bool:
    """Return True if MAX's GPU runtime supports this accelerator.

    MAX only supports certain datacenter GPUs (A10, A100, H100, etc.).
    Consumer GPUs (RTX 30xx/40xx) and other unsupported devices raise when
    queried for architecture_name or fail kernel execution with
    CUDA_ERROR_ILLEGAL_ADDRESS.  We probe here to fail fast and cleanly.
    """
    try:
        _ = acc.architecture_name
        return True
    except Exception:
        return False


class GraphCompiler:
    """Compiles a LogicalPlan into a MAX Graph.

    Key design:
    - All Filter nodes are evaluated eagerly in PyArrow *before* the graph is built
      (_strip_filters). This correctly removes rows and makes count() trivially correct.
    - Supports: Scan, Project, Filter (pre-applied), Aggregate, Sort, Limit, Distinct.
    - Sort/Limit/Distinct are stored as post-ops and applied after graph execution.
    - device: "cpu" | "gpu" | "auto"
        "cpu"  -- always use CPU (default)
        "gpu"  -- require a MAX-compatible GPU; raises RuntimeError if unavailable or
                 if the GPU is not supported by the installed MAX version.
        "auto" -- use GPU if MAX-compatible, else fall back to CPU silently
    """

    def __init__(self, device: str = "cpu"):
        # ── device resolution ────────────────────────────────────────────
        self._has_gpu = False
        self._gpu_driver = None
        resolved = device.lower()
        self._device_arg = resolved   # kept for subclass auto-switching

        if resolved == "gpu":
            if driver.accelerator_count() == 0:
                raise RuntimeError(
                    "device='gpu' requested but no GPU/accelerator is available"
                )
            acc = driver.Accelerator()
            if not _is_gpu_max_compatible(acc):
                raise RuntimeError(
                    f"device='gpu' requested but the available GPU is not supported "
                    f"by this version of MAX. MAX supports datacenter GPUs (A10, A100, "
                    f"H100, etc.). Consumer GPUs (RTX 30xx/40xx) are not currently "
                    f"supported. Use device='cpu' instead."
                )
            self._gpu_driver = acc
            self._driver_device = self._gpu_driver
            self._device_ref = DeviceRef.GPU(0)
            self._has_gpu = True
            self._session_device = "gpu"
        elif resolved == "auto":
            if driver.accelerator_count() > 0:
                try:
                    acc = driver.Accelerator()
                    if _is_gpu_max_compatible(acc):
                        self._gpu_driver = acc
                        self._driver_device = self._gpu_driver
                        self._device_ref = DeviceRef.GPU(0)
                        self._has_gpu = True
                        self._session_device = "gpu"
                    else:
                        # GPU present but not supported by MAX -- silently use CPU
                        self._driver_device = driver.CPU()
                        self._device_ref = DeviceRef.CPU()
                        self._session_device = "cpu"
                except Exception:
                    self._driver_device = driver.CPU()
                    self._device_ref = DeviceRef.CPU()
                    self._session_device = "cpu"
            else:
                self._driver_device = driver.CPU()
                self._device_ref = DeviceRef.CPU()
                self._session_device = "cpu"
        else:  # "cpu" or any unrecognised string
            self._driver_device = driver.CPU()
            self._device_ref = DeviceRef.CPU()
            self._session_device = "cpu"

        self.session = engine.InferenceSession(devices=[self._driver_device])

    # ── Post-op extraction ───────────────────────────────────────────
    @classmethod
    def _extract_post_ops(cls, plan):
        """Peel off Sort/Limit/Distinct from the top of the plan tree.

        Returns (core_plan, post_ops) where post_ops is a list of
        nodes in application order (innermost first).
        """
        post_ops = []
        while isinstance(plan, (Sort, Limit, Distinct)):
            post_ops.append(plan)
            plan = plan.input
        post_ops.reverse()  # innermost first = correct application order
        return plan, post_ops

    @classmethod
    def _apply_post_ops(cls, table: pa.Table, post_ops: list) -> pa.Table:
        """Apply Sort/Limit/Distinct operations on the Arrow result table.

        This is a fallback for the basic GraphCompiler. The CustomOpsCompiler
        overrides this to use MAX Graph ops + Mojo kernels instead.
        """
        for node in post_ops:
            if isinstance(node, Sort):
                sort_keys = [(e.args[0], "descending" if d else "ascending")
                             for e, d in zip(node.by, node.descending)]
                idx = pc.sort_indices(table, sort_keys=sort_keys)
                table = table.take(idx)
            elif isinstance(node, Limit):
                table = table.slice(0, node.n)
            elif isinstance(node, Distinct):
                cols = node.subset or table.column_names
                table = table.group_by(cols).aggregate([])
        return table

    def compile_and_run(self, plan) -> pa.Table:
        """Apply filters eagerly in PyArrow, then compile the rest to a MAX Graph."""
        # Peel off Sort/Limit/Distinct from top of plan
        core_plan, post_ops = self._extract_post_ops(plan)

        core_plan = self._strip_filters(core_plan)
        scan = self._find_scan(core_plan)
        col_names = scan.table.column_names
        col_arrays = {name: np.array(scan.table[name]) for name in col_names}

        input_types = [
            TensorType(_max_dtype(col_arrays[n]), list(col_arrays[n].shape), self._device_ref)
            for n in col_names
        ]

        graph = Graph(name="mxframe_query", input_types=input_types)
        with graph:
            input_nodes = {name: graph.inputs[i] for i, name in enumerate(col_names)}
            result_nodes = self._visit_plan(core_plan, input_nodes)
            graph.output(*result_nodes.values())

        model = self.session.load(graph)
        output_vals = model.execute(*[col_arrays[n] for n in col_names])
        result_names = list(result_nodes.keys())
        arrays = [pa.array(t.to_numpy()) for t in output_vals]
        result_table = pa.Table.from_arrays(arrays, names=result_names)

        # Apply Sort/Limit/Distinct as post-ops
        if post_ops:
            result_table = self._apply_post_ops(result_table, post_ops)

        return result_table

    @staticmethod
    def _eval_predicate(expr, table: pa.Table):
        """Evaluate a filter predicate against a PyArrow table."""
        op, args = expr.op, expr.args
        if op == "col":
            arr = table.column(args[0])
            return arr.combine_chunks() if isinstance(arr, pa.ChunkedArray) else arr
        if op == "lit":
            return args[0]
        if op in _PC_CMP_OPS:
            lhs = GraphCompiler._eval_predicate(args[0], table)
            rhs = GraphCompiler._eval_predicate(args[1], table)
            return _PC_CMP_OPS[op](lhs, rhs)
        raise NotImplementedError(f"Cannot evaluate predicate op '{op}' in PyArrow.")

    @classmethod
    def _strip_filters(cls, plan):
        """Recursively remove all Filter nodes, applying them eagerly to the Scan."""
        if isinstance(plan, Scan): return plan
        if isinstance(plan, Join):
            return Join(cls._strip_filters(plan.left), cls._strip_filters(plan.right),
                        plan.left_on, plan.right_on, plan.how)
        if isinstance(plan, Filter):
            clean_inner = cls._strip_filters(plan.input)
            scan = cls._find_scan_static(clean_inner)
            mask = cls._eval_predicate(plan.predicate, scan.table)
            filtered_scan = Scan(scan.table.filter(mask))
            return cls._replace_scan(clean_inner, filtered_scan)
        if isinstance(plan, Project):
            return Project(cls._strip_filters(plan.input), plan.exprs)
        if isinstance(plan, Aggregate):
            return Aggregate(cls._strip_filters(plan.input), plan.group_by, plan.aggs)
        if isinstance(plan, Sort):
            return Sort(cls._strip_filters(plan.input), plan.by, plan.descending)
        if isinstance(plan, Limit):
            return Limit(cls._strip_filters(plan.input), plan.n)
        if isinstance(plan, Distinct):
            return Distinct(cls._strip_filters(plan.input), plan.subset)
        if hasattr(plan, "input"):
            plan.input = cls._strip_filters(plan.input)
        return plan

    @classmethod
    def _replace_scan(cls, plan, new_scan):
        """Replace the Scan leaf in a plan tree."""
        if isinstance(plan, Scan): return new_scan
        if isinstance(plan, Join):
            raise ValueError("_replace_scan: Join should have been materialized before this point")
        if isinstance(plan, Project): return Project(cls._replace_scan(plan.input, new_scan), plan.exprs)
        if isinstance(plan, Aggregate): return Aggregate(cls._replace_scan(plan.input, new_scan), plan.group_by, plan.aggs)
        if isinstance(plan, Sort): return Sort(cls._replace_scan(plan.input, new_scan), plan.by, plan.descending)
        if isinstance(plan, Limit): return Limit(cls._replace_scan(plan.input, new_scan), plan.n)
        if isinstance(plan, Distinct): return Distinct(cls._replace_scan(plan.input, new_scan), plan.subset)
        if hasattr(plan, "input"): plan.input = cls._replace_scan(plan.input, new_scan)
        return plan

    @staticmethod
    def _find_scan_static(plan):
        if isinstance(plan, Scan): return plan
        if isinstance(plan, Join):
            raise ValueError("_find_scan: Join should have been materialized before this point")
        if hasattr(plan, "input"): return GraphCompiler._find_scan_static(plan.input)
        raise ValueError(f"No Scan found in plan: {type(plan)}")

    def _find_scan(self, plan): return self._find_scan_static(plan)

    def _visit_plan(self, plan, nodes):
        if isinstance(plan, Scan): return nodes
        elif isinstance(plan, Join):
            raise NotImplementedError("Join must be materialized before graph compilation")
        elif isinstance(plan, Project): return self._visit_project(plan, nodes)
        elif isinstance(plan, Filter): return self._visit_filter(plan, nodes)
        elif isinstance(plan, Aggregate): return self._visit_aggregate(plan, nodes)
        raise NotImplementedError(f"Unsupported plan node: {type(plan)}")

    def _visit_project(self, plan, nodes):
        upstream = self._visit_plan(plan.input, nodes)
        out = {}
        for i, expr in enumerate(plan.exprs):
            name = expr._alias or f"col_{i}"
            out[name] = self._visit_expr(expr, upstream)
        return out

    def _visit_filter(self, plan, nodes):
        upstream = self._visit_plan(plan.input, nodes)
        mask = self._visit_expr(plan.predicate, upstream)
        filtered = {}
        for name, node in upstream.items():
            filtered[name] = ops.mul(node, ops.cast(mask, node.type.dtype))
        return filtered

    def _visit_aggregate(self, plan, nodes):
        """Global (non-grouped) aggregation via built-in MAX ops."""
        upstream = self._visit_plan(plan.input, nodes)
        out = {}
        for i, expr in enumerate(plan.aggs):
            name = expr._alias or f"agg_{i}"
            out[name] = self._visit_expr(expr, upstream)
        return out

    def _visit_expr(self, expr, nodes):
        """Translate an Expr tree into MAX graph ops."""
        op, args = expr.op, expr.args
        if op == "col": return nodes[args[0]]
        if op == "lit":
            val = args[0]
            if isinstance(val, int): return ops.constant(val, dtype=DType.int64, device=self._device_ref)
            elif isinstance(val, float): return ops.constant(val, dtype=DType.float64, device=self._device_ref)
            return ops.constant(val, dtype=DType.float32, device=self._device_ref)
        if op == "add": return ops.add(self._visit_expr(args[0], nodes), self._visit_expr(args[1], nodes))
        if op == "sub": return ops.sub(self._visit_expr(args[0], nodes), self._visit_expr(args[1], nodes))
        if op == "mul": return ops.mul(self._visit_expr(args[0], nodes), self._visit_expr(args[1], nodes))
        if op == "div": return ops.div(self._visit_expr(args[0], nodes), self._visit_expr(args[1], nodes))
        if op in _CMP_OPS:
            return _CMP_OPS[op](self._visit_expr(args[0], nodes), self._visit_expr(args[1], nodes))
        if op == "sum": return ops.sum(self._visit_expr(args[0], nodes), axis=0)
        if op == "min": return ops.min(self._visit_expr(args[0], nodes), axis=0)
        if op == "max": return ops.max(self._visit_expr(args[0], nodes), axis=0)
        if op == "mean": return ops.mean(self._visit_expr(args[0], nodes), axis=0)
        if op == "count":
            col_node = self._visit_expr(args[0], nodes)
            n = col_node.type.shape[0]
            return ops.constant(np.array([int(n)], dtype=np.int64), dtype=DType.int64, device=self._device_ref)
        raise NotImplementedError(f"Unsupported expression op: '{op}'")


## Tests 🧪

Let's verify that our compiler can translate a simple plan and execute it.

In [ ]:
import pyarrow as pa
from mxframe.lazy_expr import col, lit
from mxframe.lazy_frame import LazyFrame, Scan

# ── Test 1: Simple projection ──
table = pa.table({'a': [1, 2, 3], 'b': [4, 5, 6]})
lf = LazyFrame(Scan(table))
result = GraphCompiler().compile_and_run(lf.select(col('a') + lit(10)).plan)
assert result.column(0).to_pylist() == [11, 12, 13], f'Projection failed: {result}'
print('✅ Test 1 passed: projection')

# ── Test 2: Global sum aggregation ──
lf2 = LazyFrame(Scan(pa.table({'x': [1.0, 2.0, 3.0, 4.0]})))
result2 = GraphCompiler().compile_and_run(
    lf2.groupby().agg(col('x').sum().alias('total')).plan
)
assert result2.column('total').to_pylist()[0] == 10.0, f'Sum failed: {result2}'
print('✅ Test 2 passed: global sum aggregation')

# ── Test 3: Filter REMOVES rows (not zeros them) ──
table3 = pa.table({'a': [1, 2, 3, 4, 5], 'b': [10.0, 20.0, 30.0, 40.0, 50.0]})
lf3 = LazyFrame(Scan(table3))
result3 = GraphCompiler().compile_and_run(lf3.filter(col('a') > lit(2)).plan)
assert result3.num_rows == 3, f'Filter should produce 3 rows, got {result3.num_rows}'
assert result3.column('a').to_pylist() == [3, 4, 5], f'Wrong rows: {result3.column("a").to_pylist()}'
print('✅ Test 3 passed: filter removes rows (not masks them)')

# ── Test 4: mean() is correct after filter ──
# Old bug: mask-multiply gave mean([0,0,30,40,50])/5 = 24.0
# Correct:  mean([30.0, 40.0, 50.0]) = 40.0
result4 = GraphCompiler().compile_and_run(
    lf3.filter(col('a') > lit(2)).groupby().agg(col('b').mean().alias('avg_b')).plan
)
avg_b = result4.column('avg_b').to_pylist()[0]
assert abs(avg_b - 40.0) < 1e-6, f'Mean after filter should be 40.0, got {avg_b} (old bug was 24.0)'
print('✅ Test 4 passed: mean() correct after filter (old bug was 24.0)')

# ── Test 5: count() global ──
table5 = pa.table({'v': [10, 20, 30, 40, 50]})
lf5 = LazyFrame(Scan(table5))
result5 = GraphCompiler().compile_and_run(
    lf5.groupby().agg(col('v').count().alias('n')).plan
)
assert result5.column('n').to_pylist()[0] == 5, f'Count should be 5, got {result5.column("n").to_pylist()[0]}'
print('✅ Test 5 passed: count() global')

# ── Test 6: count() after filter ──
result6 = GraphCompiler().compile_and_run(
    lf5.filter(col('v') > lit(25)).groupby().agg(col('v').count().alias('n')).plan
)
assert result6.column('n').to_pylist()[0] == 3, f'Count after filter should be 3, got {result6.column("n").to_pylist()[0]}'
print('✅ Test 6 passed: count() after filter')

print('\nAll GraphCompiler tests passed! ✅')


All GraphCompiler tests passed! ✅
